# AgriSwin-MoE: cross-class session erasure

PlantVillage's filenames record which photo session produced each image, and
31 of its 38 classes come from a single session. Class and session are
therefore the same signal in training, which is what the field collapse
exposes. This notebook estimates a session's signature in the seven classes
where session and class vary independently, erases it from all thirty-eight,
and scores the result on field photographs.

Nothing is trained here. A frozen ImageNet Swin-V2-T turns images into
vectors once, and everything after that is linear algebra, so the whole
notebook is minutes rather than hours and spends only a few compute units.

**Runtime > Change runtime type > L4 GPU > Save**, then **Runtime > Run all**.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Connect Google Drive. The image cache lives there, so this runtime does
#    not re-download PlantVillage.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Get the latest code.
import os
REPO = "/content/agriswin-moe"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --rebase
else:
    !git clone https://github.com/amritaj312-max/agriswin-moe.git {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Prepare this runtime: exact packages, images verified byte for byte.
import importlib
import sys
sys.path.insert(0, f"{REPO}/notebooks")
import colab_runtime
importlib.reload(colab_runtime)
runtime = colab_runtime.setup()

## The session table

What the filenames give us, before any model runs: how many sessions there
are, how many classes each covers, and the seven classes in which a session's
signature can be told apart from the disease it photographed.

In [ ]:
# 5. The confound, in numbers
!python ml_pipeline/src/data/sources.py

## Cutting the leaf out

A session is readable from the bench it was shot on, which would make this
trivial and useless. So the leaf is cut out with PlantVillage's own masks and
dropped on a flat random colour: whatever signature survives that is carried
by leaf pixels, which is the kind Noyan (2022) found outlives background
removal. This fetches the masks (about 1 GB; the colour images already here
are not downloaded again).

In [ ]:
# 6. PlantVillage's background-removed copies
!python ml_pipeline/src/data/download.py --segmented

In [ ]:
# 7. Cache frozen features: PlantVillage as shipped, PlantVillage as
#    leaf-on-colour, and the field set whole and cropped to its leaf box.
#    A few minutes on an L4, and the only GPU work in this notebook.
!python ml_pipeline/src/debias/features.py

## The experiment

`conditional` is the method: the signature is estimated only inside classes
photographed by two or more sessions, so a class difference can never enter
it. `marginal` is the baseline it has to beat: session means taken over the
whole dataset, which in this data are nearly the class means, so erasing them
should erase the labels. `none` is the frozen-probe baseline.

The row to watch is the class-signal check. Inside session RS, which
photographed fifteen classes, every image shares a session, so any class
signal there is class signal. An erasure that costs accuracy there took the
label with it.

In [ ]:
# 8. Estimate, erase, score
!python ml_pipeline/src/debias/erasure.py --k 1 2 4 8

In [ ]:
# 9. Save: commit and push the result (copied to Drive instead without a token)
colab_runtime.commit_and_push(
    ["results/erasure.json"],
    "Cross-class session erasure: frozen-probe prototype on " + runtime["gpu"],
    runtime["can_push"])